# Access to Opportunity
## Neighbourhood transport accessibility in Leeds, Bradford and Wakefield

**Main question:** Which neighbourhoods combine weak modelled public-transport connectivity, many households without a car and high income deprivation?

Leeds and Bradford are the main study; Wakefield is the comparison. We use all eligible LSOA 2021 areas in the full districts. This notebook explores **official observed/modelled data**, not demonstration data. Start with `reports/feasibility.md`; run the acquisition and build commands in the README before executing this notebook.

Dates differ: Census 21 March 2021; DfT Q4 2024 label, 2025 release, mixed model inputs; income mainly March 2024 with mid-2022 denominators, 2025 release. This is descriptive analysis, not causal inference.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import duckdb
import plotly.express as px

ROOT = Path.cwd()
if not (ROOT / 'config/study.yml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'config/study.yml').exists(), 'Run from the project root or notebooks directory'
sys.path.insert(0, str(ROOT))
from opportunity.analysis import prioritise
from opportunity.visuals import distribution, association_plot

data = pd.read_parquet(ROOT / 'data/processed/neighbourhoods.parquet')
quality = json.loads((ROOT / 'reports/data_quality.json').read_text())
print(f'{len(data):,} LSOAs; {data.priority_eligible.sum():,} with complete core indicators')
data.groupby(['district_name', 'study_role']).size().rename('LSOAs').to_frame()

1,015 LSOAs; 1,015 with complete core indicators


,,LSOAs
district_name,study_role,
Bradford,main,312
Leeds,main,488
Wakefield,comparison,215


### 1. Do the files actually answer our question?

Inspect the actual workbook sheet names and header—not just the publication title. The primary field is `Overall (public transport)`, combining purposes. It must not be called employment accessibility. The workbook's `Business (public transport)` label is retained but not interpreted as employment without clarification. DfT's `Overall` excludes driving. The 0–100 scores are not travel minutes, reachable-job counts or percentile ranks.

In [2]:
inspection = json.loads((ROOT / 'metadata/dft_workbook_inspection.json').read_text())
display(pd.DataFrame([{'sheet': s['name'], 'nonempty_rows_including_headers': s['rows']} for s in inspection['sheets']]))
lsoa_sheet = next(s for s in inspection['sheets'] if s['name'] == 'LSOA')
display(pd.Series(lsoa_sheet['preview'][2], name='Actual LSOA header'))

,sheet,nonempty_rows_including_headers
0,Metadata,41
1,OA,188883
2,LSOA,35675
3,LAD,334
4,RGN,13


0                                     LSOA21CD
1                         Employment (walking)
2                          Education (walking)
3                         Healthcare (walking)
4              Leisure and Community (walking)
5                           Shopping (walking)
6                        Residential (walking)
7                            Overall (walking)
8                         Employment (cycling)
9                          Education (cycling)
10                        Healthcare (cycling)
11             Leisure and Community (cycling)
12                          Shopping (cycling)
13                       Residential (cycling)
14                           Overall (cycling)
15                 Business (public transport)
16                Education (public transport)
17               Healthcare (public transport)
18    Leisure and Community (public transport)
19                 Shopping (public transport)
20              Residential (public transport)
21           

### 2. Are geography and denominators valid?

Joins are one-to-one and left-join onto an official lookup spine. A missing record remains missing. We do not directly join 2011 and 2021 codes. The unemployment rate includes active full-time students; it uses economically active age-16+ residents, while the employment and activity percentages use all age-16+ residents. TS045 uses households, not people.

In [3]:
display(pd.DataFrame(quality['joins'])[['table', 'source_rows', 'matched', 'unmatched_codes', 'cardinality']])
assert data.lsoa_code.is_unique
assert len(data) == quality['boundary_count']
np.testing.assert_allclose(data.no_car_pct, 100 * data.households_no_car / data.households_total)
np.testing.assert_allclose(data.unemployment_pct, 100 * data.unemployed_16plus / data.active_16plus)
data[['lsoa_name', 'households_no_car', 'households_total', 'no_car_pct', 'adults_16plus', 'active_16plus', 'unemployed_16plus', 'unemployment_pct']].head()

,table,source_rows,matched,unmatched_codes,cardinality
0,access,35672,1015,[],one_to_one
1,cars,35672,1015,[],one_to_one
2,activity,35672,1015,[],one_to_one
3,qualifications,35672,1015,[],one_to_one
4,population,35672,1015,[],one_to_one
5,density,35672,1015,[],one_to_one
6,age,35672,1015,[],one_to_one
7,income,33755,1015,[],one_to_one


,lsoa_name,households_no_car,households_total,no_car_pct,adults_16plus,active_16plus,unemployed_16plus,unemployment_pct
0,Bradford 016A,158,824,19.174757,1419,847,47,5.548996
1,Bradford 016B,277,841,32.936980,1426,804,45,5.597015
2,Bradford 018A,33,585,5.641026,1266,668,9,1.347305
3,Bradford 016C,289,937,30.843116,1528,986,51,5.172414
4,Bradford 016D,106,674,15.727003,1258,765,30,3.921569


### 3. How do distributions differ within and between districts?

The boxplots give each LSOA equal weight. Do not reduce a district to its centre or its average. Then use SQL to distinguish a population-weighted connectivity mean from the unweighted neighbourhood mean, and the household-level no-car rate from an average of neighbourhood percentages.

In [4]:
distribution(data, 'access_pt').show()
with duckdb.connect(str(ROOT / 'data/processed/opportunity.duckdb'), read_only=True) as con:
    summary = con.execute('''
        SELECT district_name, neighbourhoods,
               median_access, p10_access, p90_access,
               unweighted_mean_access, population_weighted_mean_access,
               household_no_car_pct, mean_neighbourhood_no_car_pct
        FROM district_summary ORDER BY median_access DESC
    ''').df()
summary.round(2)

,district_name,neighbourhoods,median_access,p10_access,p90_access,unweighted_mean_access,population_weighted_mean_access,household_no_car_pct,mean_neighbourhood_no_car_pct
0,Leeds,488,73.59,61.17,81.98,72.28,72.63,28.72,28.13
1,Bradford,312,71.60,59.53,80.14,70.53,70.89,27.59,27.61
2,Wakefield,215,64.18,51.64,73.99,63.41,63.53,22.82,22.67


### 4. Do high need and weak access necessarily coincide?

Every point below is an LSOA; dot size represents income deprivation. Inspect the observed direction instead of expecting poorer areas always to have lower connectivity. Correlations are area-level descriptions; spatial clustering and omitted factors prevent a causal interpretation. Read the within-district results as well as the pooled result.

In [5]:
association_plot(data).show()
pd.read_csv(ROOT / 'data/processed/associations.csv').round(3)

,district,variable,complete_pairs,spearman_rho
0,All study districts,no_car_pct,1015,0.697
1,All study districts,income_pct,1015,0.424
2,All study districts,unemployment_pct,1015,0.601
3,All study districts,residents_per_km2,1015,0.692
4,Bradford,no_car_pct,312,0.700
5,Bradford,income_pct,312,0.643
6,Bradford,unemployment_pct,312,0.731
7,Bradford,residents_per_km2,312,0.659
8,Leeds,no_car_pct,488,0.770
9,Leeds,income_pct,488,0.427


### 5. Which neighbourhoods meet a transparent priority rule?

Baseline: all three adverse percentile standings must be at least 0.75 in the pooled study. Ties share a mid-percentile. A separate equal-weight composite averages those standings. An area can have a high composite without being in every adverse quartile.

This is a screening definition, not an official deprivation index, funding decision, probability or individual diagnosis.

In [6]:
columns = ['lsoa_code', 'lsoa_name', 'district_name', 'access_pt', 'no_car_pct', 'income_pct', 'priority_score', 'triple_priority']
display(data.loc[data.triple_priority.fillna(False), columns].sort_values('priority_score', ascending=False))
display(data.nlargest(10, 'priority_score')[columns])

,lsoa_code,lsoa_name,district_name,access_pt,no_car_pct,income_pct,priority_score,triple_priority
804,E01011811,Wakefield 011E,Wakefield,56.27,39.823009,58.8,88.013136,True
785,E01011792,Wakefield 039D,Wakefield,59.71,39.741519,66.3,87.471264,True
147,E01010718,Bradford 012C,Bradford,60.21,44.171779,50.2,85.073892,True


,lsoa_code,lsoa_name,district_name,access_pt,no_car_pct,income_pct,priority_score,triple_priority
804,E01011811,Wakefield 011E,Wakefield,56.27,39.823009,58.8,88.013136,True
785,E01011792,Wakefield 039D,Wakefield,59.71,39.741519,66.3,87.471264,True
147,E01010718,Bradford 012C,Bradford,60.21,44.171779,50.2,85.073892,True
773,E01011780,Wakefield 027B,Wakefield,66.05,44.958482,61.6,84.203612,False
67,E01010635,Bradford 047C,Bradford,67.80,49.240122,61.2,83.497537,False
44,E01010612,Bradford 052A,Bradford,70.34,54.637437,77.1,83.169130,False
742,E01011745,Wakefield 001D,Wakefield,57.76,35.507246,52.7,82.939245,False
744,E01011747,Wakefield 003C,Wakefield,63.04,36.016371,61.7,82.758621,False
248,E01010821,Bradford 052D,Bradford,68.12,45.419847,66.9,82.463054,False
631,E01011624,Leeds 072D,Leeds,69.57,46.556474,59.6,80.328407,False


### 6. How much do the priorities depend on our choices?

Compare adverse fifths, quartiles and thirds; equal, access-emphasis and income-emphasis weights; pooled, within-district and Leeds/Bradford reference groups. Weights affect the composite, not the all-three-tail rule. Jaccard is overlap divided by union; undefined when both sets are empty. Frequencies across scenarios describe sensitivity, not statistical confidence.

In [7]:
scenarios = pd.read_csv(ROOT / 'data/processed/sensitivity.csv')
display(scenarios.loc[scenarios.weights.eq('equal')].round(3))
display(data.nlargest(10, 'priority_score')[['lsoa_name', 'triple_frequency', 'top_score_frequency', 'best_score_rank', 'worst_score_rank']])
baseline = set(data.loc[data.triple_priority.fillna(False), 'lsoa_code'])
alternative = set(data.loc[data.sustainable_triple_priority.fillna(False), 'lsoa_code'])
print('Primary all-tail areas:', len(baseline), 'Sustainable-mode alternative:', len(alternative))
print('Same membership:', baseline == alternative)

,scenario,comparison_group,tail_fraction,weights,eligible,triple_count,top_score_count,triple_jaccard_vs_baseline,score_band_jaccard_vs_baseline
0,pooled|0.200000|equal,pooled,0.200,equal,1015,1,203,0.333,0.799
3,pooled|0.250000|equal,pooled,0.250,equal,1015,3,254,1.000,1.000
6,pooled|0.333333|equal,pooled,0.333,equal,1015,19,339,0.158,0.749
9,district|0.200000|equal,district,0.200,equal,1015,1,203,0.333,0.693
12,district|0.250000|equal,district,0.250,equal,1015,3,254,0.200,0.728
15,district|0.333333|equal,district,0.333,equal,1015,16,339,0.188,0.724
18,main_reference|0.200000|equal,main_reference,0.200,equal,1015,1,203,0.333,0.799
21,main_reference|0.250000|equal,main_reference,0.250,equal,1015,4,255,0.750,0.921
24,main_reference|0.333333|equal,main_reference,0.333,equal,1015,24,338,0.125,0.751


,lsoa_name,triple_frequency,top_score_frequency,best_score_rank,worst_score_rank
804,Wakefield 011E,0.777778,1.0,1.0,3.0
785,Wakefield 039D,0.555556,1.0,1.0,5.0
147,Bradford 012C,0.777778,1.0,3.0,35.0
773,Wakefield 027B,0.333333,1.0,4.0,70.0
67,Bradford 047C,0.222222,1.0,5.0,13.0
44,Bradford 052A,0.0,1.0,2.0,27.0
742,Wakefield 001D,0.444444,1.0,3.0,15.0
744,Wakefield 003C,0.222222,1.0,5.0,32.0
248,Bradford 052D,0.111111,1.0,6.0,17.0
631,Leeds 072D,0.222222,1.0,6.0,34.0


Primary all-tail areas: 3 Sustainable-mode alternative: 3
Same membership: True


### 7. What if we compare similar neighbourhoods?

Form joint tercile groups of income need, population density, age 0–14 share and age 65+ share. A supported group needs at least five areas across two districts. Show covariate ranges: the same bin does not mean identical characteristics. The group median includes the area itself. These are coarse descriptive peers, not matched causal controls.

In [8]:
supported = data.loc[data.peer_supported.fillna(False)]
print(f'Peer-supported areas: {len(supported)} / {len(data)}')
display(supported.nsmallest(10, 'peer_access_gap')[['lsoa_name', 'district_name', 'access_pt', 'peer_group', 'peer_count', 'peer_districts', 'peer_median_access', 'peer_access_gap']])
groups = pd.read_csv(ROOT / 'data/processed/peer_groups.csv')
display(groups.head())

Peer-supported areas: 908 / 1015


,lsoa_name,district_name,access_pt,peer_group,peer_count,peer_districts,peer_median_access,peer_access_gap
842,Wakefield 034E,Wakefield,31.86,0/0/0/2,86,3,61.160,-29.300
282,Bradford 023E,Bradford,37.23,0/0/0/1,9,3,66.400,-29.170
850,Wakefield 034F,Wakefield,33.21,0/0/0/2,86,3,61.160,-27.950
847,Wakefield 041A,Wakefield,44.37,1/1/2/0,11,3,70.650,-26.280
848,Wakefield 041B,Wakefield,48.68,1/0/0/1,7,3,72.890,-24.210
930,Wakefield 035D,Wakefield,43.60,0/0/0/1,9,3,66.400,-22.800
927,Wakefield 038A,Wakefield,39.99,0/0/1/1,25,3,62.630,-22.640
920,Wakefield 035A,Wakefield,48.38,1/1/0/2,30,3,69.865,-21.485
923,Wakefield 035B,Wakefield,44.78,1/0/0/2,31,3,64.920,-20.140
341,Leeds 022A,Leeds,40.96,0/0/1/2,40,2,60.875,-19.915


,peer_group,peer_count,peer_districts,peer_median_access,peer_min_access,peer_max_access,income_rate_min,income_rate_max,residents_per_km2_min,residents_per_km2_max,age_0_14_pct_min,age_0_14_pct_max,age_65plus_pct_min,age_65plus_pct_max
0,0/0/0/0,2,2,76.48,67.29,85.67,0.072,0.088,1037.2,1375.6,4.389313,15.377932,2.290076,12.988705
1,0/0/0/1,9,3,66.40,37.23,74.25,0.045,0.130,167.9,1872.4,11.241507,16.266667,13.435528,19.926471
2,0/0/0/2,86,3,61.16,31.86,75.66,0.023,0.132,46.5,2373.9,10.377963,16.254876,20.561178,39.275148
3,0/0/1/0,2,1,68.93,67.48,70.38,0.060,0.120,905.0,1107.2,19.132150,19.299500,9.471051,10.650888
4,0/0/1/1,25,3,62.63,39.99,80.62,0.028,0.135,86.3,2339.4,16.602810,20.314806,13.323045,20.084694


### 8. What remains uncertain?

- Census 2021 reflects pandemic conditions and disclosure control; source years do not align.
- A neighbourhood average does not describe every resident; household car availability does not guarantee personal access.
- DfT assumes timetabled services. Reliability, fares, disability access and actual opportunities vary.
- Existing jobs are not vacancies; the score is not a count reachable within 30/45/60 minutes.
- Geography affects results; large rural polygons can dominate a map visually.
- Source model uncertainty is not provided, so we do not manufacture confidence intervals. The sensitivity grid measures our analytical choices only.
- DfT's workbook licence says TBA although the guidance states OGL. Preserve the discrepancy and attribution; raw files are not committed.

**Interview reflection:** Explain one decision you changed after inspecting the real files. Explain why the small three-tail intersection is a useful result rather than a reason to change the threshold. Choose an area from the complete export, inspect all its indicators and peer group, and describe what further evidence you would collect.

See `reports/findings.md` for computed conclusions and `docs/interview.md` for practice questions.